# Episode 21: Agent UI with AG-UI

Your agents work — but so far only from a notebook. Real users need a **web interface**.

**In this episode you'll build:** an agent exposed over the **AG-UI protocol**, ready for a streaming web frontend.

## What AG-UI is

AG-UI is a standard protocol between an agent backend and a web frontend. Instead of inventing your own REST contract, you speak AG-UI and any AG-UI client — **CopilotKit**, a custom React app — can connect and get:

- streaming text, token by token
- tool calls rendered as they happen
- shared state and human-input checkpoints

AG2 Beta wraps any `Agent` for AG-UI with one class: `AGUIStream`.

## Install

The AG-UI integration is an optional extra.

In [1]:
!pip install "ag2[ag-ui]" uvicorn -q

## Build the AG-UI endpoint

`AGUIStream(agent)` wraps the agent; `.build_asgi()` returns an ASGI app you mount on a route. That's the whole backend — three lines.

In [2]:
from dotenv import load_dotenv

load_dotenv()

from autogen.beta import Agent
from autogen.beta.config import OpenAIConfig
from autogen.beta.ag_ui import AGUIStream
from starlette.applications import Starlette

config = OpenAIConfig(model="gpt-4.1-mini", temperature=0)

agent = Agent("support_bot",
              prompt="You help users with billing questions. Be concise.",
              config=config)

stream = AGUIStream(agent)              # wrap the agent for AG-UI
app = Starlette()
app.mount("/chat", stream.build_asgi())  # mount it as an ASGI endpoint

print("AGUIStream built:", type(stream).__name__)
print("ASGI app mounted at /chat:", type(app).__name__)
print("routes:", [r.path for r in app.routes])


AGUIStream built: AGUIStream
ASGI app mounted at /chat: Starlette
routes: ['/chat']


## Run it as a server

The cell above *builds* the app. To actually serve it you run it with `uvicorn` — and because a server runs until you stop it, that happens in a **terminal, not this notebook**. Save the following as `ag_ui_server.py`, run `python ag_ui_server.py`, then point an AG-UI client at `http://localhost:8000/chat`.

```python
# ag_ui_server.py
from dotenv import load_dotenv
load_dotenv()

import uvicorn
from starlette.applications import Starlette

from autogen.beta import Agent
from autogen.beta.ag_ui import AGUIStream
from autogen.beta.config import OpenAIConfig

agent = Agent("support_bot",
              prompt="You help users with billing questions. Be concise.",
              config=OpenAIConfig(model="gpt-4.1-mini"))

app = Starlette()
app.mount("/chat", AGUIStream(agent).build_asgi())

if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=8000)
```

## The frontend

Any AG-UI client connects to `/chat`. The quickest is **CopilotKit** (React):

```bash
npx copilotkit@latest create -f ag2
```

Or clone the reference starter — `github.com/ag2ai/ag2-copilotkit-starter` — which ships a Python backend and a React + CopilotKit frontend already wired together. Point its `HttpAgent` URL at your `/chat` endpoint and you have a streaming chat UI.

## Additional content

**`build_asgi()` vs `dispatch()`.** `build_asgi()` is the zero-config path. If you need custom auth or middleware, use `stream.dispatch(message, accept=...)` inside your own FastAPI route instead — same protocol, your own plumbing.

**What the protocol carries.** AG-UI isn't just text streaming — it surfaces tool-call lifecycle events and human-input checkpoints too. A frontend can render "calling `get_invoice`…" live, or pause for an approval, all over the one connection.

**When *not* to use AG-UI.** If you own both ends and want a narrow, custom contract, a plain endpoint is simpler — which is exactly the next episode.

## What's next

AG-UI is the right call for a rich, standard frontend. Episode 22 shows the opposite end — a chat UI with **no framework at all**, just HTML and `fetch`.